# Scraping Financial Data

This notebook scrapes data from:
- S&P 500 index using yfinance
- Oil prices from Hugging Face dataset (harishb00/daily-oil-prices)

Filtering for dates between July 5, 2024, and November 4, 2024.

In [2]:
# Install required libraries if not already installed
%pip install datasets pandas yfinance pandas_datareader

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Import libraries
import pandas as pd
from datasets import load_dataset
import yfinance as yf

c:\Users\pibrys\anaconda3\envs\SMWA2026\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
# Load S&P 500 data using yfinance
df_sp = yf.download('^GSPC', start='2024-07-05', end='2024-11-04')
print("S&P 500 dataset columns:", df_sp.columns.tolist())
print("Date range:", df_sp.index.min(), "to", df_sp.index.max())
print("Sample data:")
print(df_sp.head())

[*********************100%***********************]  1 of 1 completed


S&P 500 dataset columns: [('Close', '^GSPC'), ('High', '^GSPC'), ('Low', '^GSPC'), ('Open', '^GSPC'), ('Volume', '^GSPC')]
Date range: 2024-07-05 00:00:00 to 2024-11-01 00:00:00
Sample data:
Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2024-07-05  5567.189941  5570.330078  5531.629883  5537.910156  3253080000
2024-07-08  5572.850098  5583.109863  5562.509766  5572.750000  3185670000
2024-07-09  5576.979980  5590.750000  5574.569824  5584.240234  3232920000
2024-07-10  5633.910156  5635.390137  5586.439941  5591.259766  3336100000
2024-07-11  5584.540039  5642.450195  5576.529785  5635.209961  4020950000


In [20]:
# Filter S&P 500 data by date (already filtered in download)
df_sp_filtered = df_sp.reset_index()
df_sp_filtered.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
print(f"Filtered S&P 500 data: {len(df_sp_filtered)} rows")
print(df_sp_filtered.head())

Filtered S&P 500 data: 85 rows
        Date        Close         High          Low         Open      Volume
0 2024-07-05  5567.189941  5570.330078  5531.629883  5537.910156  3253080000
1 2024-07-08  5572.850098  5583.109863  5562.509766  5572.750000  3185670000
2 2024-07-09  5576.979980  5590.750000  5574.569824  5584.240234  3232920000
3 2024-07-10  5633.910156  5635.390137  5586.439941  5591.259766  3336100000
4 2024-07-11  5584.540039  5642.450195  5576.529785  5635.209961  4020950000


In [21]:
# Save S&P 500 filtered data to CSV
df_sp_filtered.to_csv('Storage/sp500_data_jul_nov_2024.csv', index=False)
print("S&P 500 data saved to Storage/sp500_data_jul_nov_2024.csv")

S&P 500 data saved to Storage/sp500_data_jul_nov_2024.csv


In [9]:
# Load Oil Prices dataset
dataset_oil = load_dataset('harishb00/daily-oil-prices')
df_oil = pd.DataFrame(dataset_oil['train'])
print("Oil Prices dataset columns:", df_oil.columns.tolist())
print("Date range:", df_oil['Date'].min(), "to", df_oil['Date'].max())
print("Sample data:")
print(df_oil.head())

Oil Prices dataset columns: ['Date', 'Price']
Date range: 1987-05-20 to 2025-01-21
Sample data:
         Date  Price
0  1987-05-20  18.63
1  1987-05-21  18.45
2  1987-05-22  18.55
3  1987-05-25  18.60
4  1987-05-26  18.63


In [10]:
# Filter Oil Prices data by date
df_oil['Date'] = pd.to_datetime(df_oil['Date'])
df_oil_filtered = df_oil[(df_oil['Date'] >= start_date) & (df_oil['Date'] <= end_date)]
print(f"Filtered Oil Prices data: {len(df_oil_filtered)} rows")
print(df_oil_filtered.head())

Filtered Oil Prices data: 86 rows
           Date  Price
9421 2024-07-05  88.66
9422 2024-07-08  87.15
9423 2024-07-09  86.48
9424 2024-07-10  86.55
9425 2024-07-11  86.49


In [14]:
# Save Oil Prices filtered data to CSV
df_oil_filtered.to_csv('Storage/oil_prices_jul_nov_2024.csv', index=False)
print("Oil Prices data saved to Storage/oil_prices_jul_nov_2024.csv")

Oil Prices data saved to Storage/oil_prices_jul_nov_2024.csv


In [6]:
# Scrape FRED data using provided API key
from pandas_datareader import data as pdr

fred_api_key = '981ac4d5555a2e7f975219ee2bc806a1'
start_date = '2024-07-05'
end_date = '2024-11-04'

# Series to get
series = {
    'DSPIC96': 'Real disposable personal income (2012 dollars)',
    'GDPC1': 'Real GDP',
    'UNRATE': 'Unemployment Rate',
    'CPIAUCSL': 'CPI Inflation (All Urban Consumers)',
    'UMCSENT': 'Consumer Sentiment'
}

fred_data = {}
for symbol, name in series.items():
    try:
        fred_data[symbol] = pdr.DataReader(symbol, 'fred', start_date, end_date, api_key=fred_api_key)
        print(f"Fetched {symbol} ({name}): {fred_data[symbol].shape[0]} rows")
    except Exception as e:
        print(f"Error fetching {symbol}: {e}")

# Combine into one dataframe
df_fred = pd.concat(fred_data.values(), axis=1)
# rename columns to avoid conflicting names
df_fred.columns = list(series.keys())

print('FRED merged data sample:')
print(df_fred.tail())

# Save to CSV
df_fred.to_csv('Storage/fred_data_jul_nov_2024.csv', index=True)
print('FRED data saved to Storage/fred_data_jul_nov_2024.csv')

Fetched DSPIC96 (Real disposable personal income (2012 dollars)): 4 rows
Fetched GDPC1 (Real GDP): 1 rows
Fetched UNRATE (Unemployment Rate): 4 rows
Fetched CPIAUCSL (CPI Inflation (All Urban Consumers)): 4 rows
Fetched UMCSENT (Consumer Sentiment): 4 rows
FRED merged data sample:
            DSPIC96      GDPC1  UNRATE  CPIAUCSL  UMCSENT
DATE                                                     
2024-08-01  17752.9        NaN     4.2   314.062     67.9
2024-09-01  17769.9        NaN     4.1   314.732     70.1
2024-10-01  17810.5  23586.542     4.1   315.631     70.5
2024-11-01  17851.4        NaN     4.2   316.528     71.8
FRED data saved to Storage/fred_data_jul_nov_2024.csv


## Summary

The notebook has successfully scraped and filtered data from the two Hugging Face datasets for the specified date range. The filtered data has been saved to CSV files in the Storage subdirectory.